# Deep RL for Personalized News Recommendation
### MC · Q-Learning · SARSA · DQN · DQN+ICM — MINDsmall Dataset
**Yogeshvar Reddy Kallam & Ajay Krishna Devulapally**
IST 597 Deep RL · Penn State Spring 2025

---

## Experiment Overview

**Dataset:** MINDsmall — 50K users, 157K sessions, 51K articles (Microsoft News)

**MDP Formulation:**
- **State:** Concat of 384-dim sentence embeddings of last 10 clicked article titles → 3,840-dim vector
- **Action:** Select from impression list (up to 20 articles)
- **Reward:** +1 if user clicked, 0 otherwise (+ ICM intrinsic reward for DQN+ICM)
- **γ:** 0.90 (tabular) / 0.99 (DQN)

**Results (K=10):**

| Algorithm | HitRate@10 | NDCG@10 |
|-----------|------------|---------|
| Monte Carlo | 0.0347 | 0.0545 |
| Q-Learning  | 0.0581 | 0.0300 |
| SARSA       | 0.0645 | 0.0321 |
| DQN         | 0.3248 | 0.2786 |
| **DQN+ICM** | **0.6240** | **0.4432** |


## 1. Data Loading & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Expected files: data/news.tsv, data/behaviors.tsv
# Download from: https://msnews.github.io

DATA_DIR = Path("data")

def load_mind(data_dir):
    news = pd.read_csv(data_dir/"news.tsv", sep="\t", header=None,
                       names=["news_id","category","subcategory","title","abstract","url","title_entities","abstract_entities"])
    behaviors = pd.read_csv(data_dir/"behaviors.tsv", sep="\t", header=None,
                            names=["impression_id","user_id","time","history","impressions"])
    news = news[["news_id","category","title"]].dropna(subset=["title"])
    behaviors["history_list"] = behaviors["history"].fillna("").str.split()
    behaviors["impression_list"] = behaviors["impressions"].str.split()
    print(f"Loaded: {len(news):,} articles, {len(behaviors):,} sessions")
    return news, behaviors

# Uncomment when data is available:
# news_df, beh_df = load_mind(DATA_DIR)
print("Data loading function defined. Place MINDsmall files in data/ directory.")
print("Download: https://msnews.github.io")


## 2. State Representation — Sentence Embeddings

In [ ]:
# Install: pip install sentence-transformers
# from sentence_transformers import SentenceTransformer
import numpy as np

MAX_HIST = 10      # last N clicked articles
EMBED_DIM = 384    # all-MiniLM-L6-v2 output dim
STATE_DIM = MAX_HIST * EMBED_DIM   # 3,840

def build_state_embedding(click_history, title_embeddings, max_hist=MAX_HIST):
    """
    Build 3,840-dim state from user click history.
    Zero-pads if history < max_hist.
    """
    embeds = [title_embeddings[nid] for nid in click_history[-max_hist:]
              if nid in title_embeddings]
    if len(embeds) == 0:
        return np.zeros(STATE_DIM, dtype=np.float32)
    # Pad to max_hist
    while len(embeds) < max_hist:
        embeds.insert(0, np.zeros(EMBED_DIM, dtype=np.float32))
    return np.concatenate(embeds[-max_hist:], axis=0).astype(np.float32)

print(f"State dimension: {STATE_DIM}  ({MAX_HIST} articles × {EMBED_DIM} dims)")
print("Embedding model: all-MiniLM-L6-v2 (sentence-transformers)")


## 3. Tabular Agents — MC, Q-Learning, SARSA

In [ ]:
from collections import defaultdict

class TabularMC:
    """Every-visit Monte Carlo Control."""
    def __init__(self, gamma=0.9, epsilon=0.1):
        self.Q = defaultdict(lambda: defaultdict(float))
        self.N = defaultdict(lambda: defaultdict(int))
        self.gamma, self.eps = gamma, epsilon

    def act(self, state, actions):
        if np.random.rand() < self.eps:
            return np.random.choice(actions)
        return max(actions, key=lambda a: self.Q[state][a])

    def update(self, episode):
        """episode: list of (state, action, reward)"""
        G = 0.0
        for s, a, r in reversed(episode):
            G = r + self.gamma * G
            self.N[s][a] += 1
            self.Q[s][a] += (G - self.Q[s][a]) / self.N[s][a]


class TabularQLearning:
    """Off-policy TD Control."""
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.Q = defaultdict(lambda: defaultdict(float))
        self.alpha, self.gamma, self.eps = alpha, gamma, epsilon

    def act(self, state, actions):
        if np.random.rand() < self.eps: return np.random.choice(actions)
        return max(actions, key=lambda a: self.Q[state][a])

    def update(self, s, a, r, s_next, actions_next):
        best_next = max(self.Q[s_next][a2] for a2 in actions_next) if actions_next else 0.0
        self.Q[s][a] += self.alpha * (r + self.gamma * best_next - self.Q[s][a])


class TabularSARSA:
    """On-policy TD Control."""
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.Q = defaultdict(lambda: defaultdict(float))
        self.alpha, self.gamma, self.eps = alpha, gamma, epsilon

    def act(self, state, actions):
        if np.random.rand() < self.eps: return np.random.choice(actions)
        return max(actions, key=lambda a: self.Q[state][a])

    def update(self, s, a, r, s_next, a_next):
        self.Q[s][a] += self.alpha * (r + self.gamma * self.Q[s_next][a_next] - self.Q[s][a])

print("✅ Tabular agents defined (MC, Q-Learning, SARSA)")


## 4. DQN Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

STATE_DIM  = 3840   # 10 × 384
MAX_IMPR   = 20     # max impression list size

class QNet(nn.Module):
    """DQN network: 3840-dim state → Q-values for each impression slot."""
    def __init__(self, state_dim=STATE_DIM, n_actions=MAX_IMPR, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 512), nn.ReLU(),
            nn.Linear(512, hidden),    nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )
    def forward(self, x):
        return self.net(x)

# Experience Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity=20_000):
        self.buf = []; self.cap = capacity
    def push(self, *transition):
        if len(self.buf) >= self.cap: self.buf.pop(0)
        self.buf.append(transition)
    def sample(self, batch_size):
        import random
        batch = random.sample(self.buf, min(batch_size, len(self.buf)))
        return list(zip(*batch))
    def __len__(self): return len(self.buf)

qnet        = QNet()
target_net  = QNet()
target_net.load_state_dict(qnet.state_dict())
print(f"QNet parameters: {sum(p.numel() for p in qnet.parameters()):,}")
print(f"Architecture: Linear(3840→512)→ReLU→Linear(512→256)→ReLU→Linear(256→{MAX_IMPR})")


## 5. DQN + ICM Architecture

In [ ]:
class ICMEncoder(nn.Module):
    """Maps state embeddings to compact feature vectors."""
    def __init__(self, state_dim=STATE_DIM, feat_dim=128):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(state_dim, 256), nn.ReLU(), nn.Linear(256, feat_dim))
    def forward(self, x): return self.enc(x)

class ICMInverseModel(nn.Module):
    """Predicts taken action from (φ(s), φ(s'))."""
    def __init__(self, feat_dim=128, n_actions=MAX_IMPR):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(feat_dim*2, 256), nn.ReLU(), nn.Linear(256, n_actions))
    def forward(self, phi_s, phi_s_next): return self.net(torch.cat([phi_s, phi_s_next], dim=-1))

class ICMForwardModel(nn.Module):
    """Predicts φ(s') from (φ(s), action)."""
    def __init__(self, feat_dim=128, n_actions=MAX_IMPR):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(feat_dim + n_actions, 256), nn.ReLU(), nn.Linear(256, feat_dim))
    def forward(self, phi_s, action_onehot): return self.net(torch.cat([phi_s, action_onehot], dim=-1))

class ICM(nn.Module):
    def __init__(self, state_dim=STATE_DIM, feat_dim=128, n_actions=MAX_IMPR,
                 reward_scale=10.0, inv_weight=1.0, fwd_weight=50.0):
        super().__init__()
        self.encoder = ICMEncoder(state_dim, feat_dim)
        self.inverse  = ICMInverseModel(feat_dim, n_actions)
        self.forward_m= ICMForwardModel(feat_dim, n_actions)
        self.n_actions  = n_actions
        self.rw_scale   = reward_scale
        self.inv_w, self.fwd_w = inv_weight, fwd_weight

    def compute_intrinsic_reward(self, s, a, s_next):
        phi_s      = self.encoder(s)
        phi_s_next = self.encoder(s_next)
        a_oh = F.one_hot(a, self.n_actions).float()
        phi_pred   = self.forward_m(phi_s, a_oh)
        r_int = self.rw_scale * 0.5 * F.mse_loss(phi_pred, phi_s_next.detach(), reduction='none').mean(-1)
        return r_int, phi_s, phi_s_next, phi_pred

    def compute_losses(self, s, a, s_next):
        r_int, phi_s, phi_s_next, phi_pred = self.compute_intrinsic_reward(s, a, s_next)
        a_pred   = self.inverse(phi_s, phi_s_next)
        loss_inv = self.inv_w * F.cross_entropy(a_pred, a)
        loss_fwd = self.fwd_w * F.mse_loss(phi_pred, phi_s_next.detach())
        return r_int, loss_inv, loss_fwd

print("✅ ICM components defined:")
print("   • ICMEncoder       — state → feature vector")
print("   • ICMInverseModel  — (φ(s), φ(s')) → predicted action")
print("   • ICMForwardModel  — (φ(s), action) → predicted φ(s')")
print("   • ICM              — full module with intrinsic reward computation")


## 6. Evaluation Metrics

In [ ]:
def hit_rate_at_k(ranked_items, relevant_item, k=10):
    """HitRate@K: 1 if relevant item in top-K, else 0."""
    return 1.0 if relevant_item in ranked_items[:k] else 0.0

def ndcg_at_k(ranked_items, relevant_item, k=10):
    """NDCG@K: normalized discounted cumulative gain."""
    if relevant_item not in ranked_items[:k]:
        return 0.0
    rank = ranked_items[:k].index(relevant_item) + 1
    return 1.0 / np.log2(rank + 1)

def mrr_at_k(ranked_items, relevant_item, k=10):
    """MRR@K: mean reciprocal rank."""
    if relevant_item not in ranked_items[:k]:
        return 0.0
    rank = ranked_items[:k].index(relevant_item) + 1
    return 1.0 / rank

def evaluate_agent(agent_fn, sessions, k=10):
    """Offline evaluation: rank items per session, compute metrics."""
    hr, ndcg, mrr = [], [], []
    for session in sessions:
        state, candidates, clicked = session
        q_vals   = agent_fn(state, candidates)
        ranked   = [candidates[i] for i in np.argsort(q_vals)[::-1]]
        hr.append(hit_rate_at_k(ranked, clicked, k))
        ndcg.append(ndcg_at_k(ranked, clicked, k))
        mrr.append(mrr_at_k(ranked, clicked, k))
    return {"HitRate@K": np.mean(hr), "NDCG@K": np.mean(ndcg), "MRR@K": np.mean(mrr)}

# ── Replicated paper results ──────────────────────────────────────────────
results = {
    "Monte Carlo":  {"HitRate@10": 0.0347, "NDCG@10": 0.0545},
    "Q-Learning":   {"HitRate@10": 0.0581, "NDCG@10": 0.0300},
    "SARSA":        {"HitRate@10": 0.0645, "NDCG@10": 0.0321},
    "DQN":          {"HitRate@10": 0.3248, "NDCG@10": 0.2786},
    "DQN + ICM":    {"HitRate@10": 0.6240, "NDCG@10": 0.4432},
}

import matplotlib.pyplot as plt
algorithms = list(results.keys())
hr_vals    = [v["HitRate@10"] for v in results.values()]
ndcg_vals  = [v["NDCG@10"]   for v in results.values()]

x = np.arange(len(algorithms)); w = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w/2, hr_vals,   w, label="HitRate@10",  color="#4C72B0")
b2 = ax.bar(x + w/2, ndcg_vals, w, label="NDCG@10",     color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(algorithms, rotation=15, ha="right")
ax.set_ylabel("Score"); ax.set_title("Algorithm Comparison — News Recommendation (K=10)")
ax.legend(); ax.bar_label(b1, fmt="%.3f", padding=2); ax.bar_label(b2, fmt="%.3f", padding=2)
plt.tight_layout(); plt.show()
